# unit commitment milp and scenarios — exercises

12 tasks that test what the cheat sheet `06_unit_commitment_milp_and_scenarios.ipynb` covers. Try each one
**before** looking at the cheat sheet.

How it works:
1. Read the task. Write your code in the cell below it, assigning the result to the
   variable the task names (usually `answer`).
2. Run the cell: the last line `qN.check()` tells you ✅ or ❌ with a short reason.
3. Stuck? Uncomment `qN.hint()` for a nudge, or `qN.solution()` to see the reference code.

In [ ]:
import sys; sys.path.append("..")
from quantlearn import load_questions
q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12 = load_questions("07_optimisation/06_unit_commitment_milp_and_scenarios")

import numpy as np
import pandas as pd
from scipy.optimize import milp, linprog, LinearConstraint, Bounds, differential_evolution, minimize

## Task 1

Solve the tiny problem as a plain LP (no integrality) with `milp`: minimise 2x + 3y subject to x + y ≥ 4.5, 0 ≤ x ≤ 3, y ≥ 0. Assign the optimal `x_lp` (array of the two variables) and `cost_lp`.

Assign the result to `x_lp`, `cost_lp`.

In [ ]:
# minimise 2x + 3y  subject to  x + y >= 4.5,  0 <= x <= 3,  y >= 0
c = np.array([2.0, 3.0])
A = np.array([[1.0, 1.0]])
constraints = LinearConstraint(A, [4.5], [np.inf])
bounds = Bounds([0, 0], [3, np.inf])

# your solution here
x_lp = ____
cost_lp = ____

q1.check()

In [ ]:
# q1.hint()
# q1.solution()

## Task 2

Now require y to be an integer (integrality = [0, 1]). Assign the MILP solution `x_mip` and its cost `cost_mip`. Why is the cost higher?

Assign the result to `x_mip`, `cost_mip`.

In [ ]:
# minimise 2x + 3y  subject to  x + y >= 4.5,  0 <= x <= 3,  y >= 0
c = np.array([2.0, 3.0])
A = np.array([[1.0, 1.0]])
constraints = LinearConstraint(A, [4.5], [np.inf])
bounds = Bounds([0, 0], [3, np.inf])

# your solution here
x_mip = ____
cost_mip = ____

q2.check()

In [ ]:
# q2.hint()
# q2.solution()

## Task 3

Make the tiny problem infeasible: keep x ≤ 3 but also cap y ≤ 1 (so x + y ≤ 4 < 4.5). Solve with milp and assign the returned `status` code (2 means infeasible).

Assign the result to `status`.

In [ ]:
# minimise 2x + 3y  subject to  x + y >= 4.5,  0 <= x <= 3,  y >= 0
c = np.array([2.0, 3.0])
A = np.array([[1.0, 1.0]])
constraints = LinearConstraint(A, [4.5], [np.inf])
bounds = Bounds([0, 0], [3, np.inf])

# your solution here
status = ____

q3.check()

In [ ]:
# q3.hint()
# q3.solution()

## Task 4

Unit commitment, 2 units × 2 hours: `c`, `A`, `lb`, `ub`, `integrality` and the variable upper bounds `hi` are built for you (read the comments). Solve with milp (lower bounds 0). Assign the optimal `cost` and the four on/off decisions `u` as an integer array in the order u_A0, u_A1, u_B0, u_B1.

Assign the result to `cost`, `u`.

In [ ]:
# Two units, two hours. Variable order:
#   p_A0, p_A1, p_B0, p_B1   production (MW)      continuous
#   u_A0, u_A1, u_B0, u_B1   on/off               binary
#   s_A0, s_A1, s_B0, s_B1   start-up indicator   binary
names = ["p_A0", "p_A1", "p_B0", "p_B1", "u_A0", "u_A1", "u_B0", "u_B1", "s_A0", "s_A1", "s_B0", "s_B1"]
col = {n: i for i, n in enumerate(names)}
demand = np.array([6.0, 11.0])
mc    = {"A": 10.0, "B": 30.0}     # marginal cost EUR/MWh
oncost = {"A": 5.0, "B": 2.0}      # cost per hour while on
startup = {"A": 20.0, "B": 10.0}   # cost of switching on (both units start OFF)
pmax = {"A": 8.0, "B": 6.0}
pmin = {"A": 2.0, "B": 1.0}

c = np.zeros(12)
for g in ["A", "B"]:
    for h in range(2):
        c[col[f"p_{g}{h}"]] = mc[g]
        c[col[f"u_{g}{h}"]] = oncost[g]
        c[col[f"s_{g}{h}"]] = startup[g]

rows, lb, ub = [], [], []
for h in range(2):                                   # demand: p_A + p_B = demand
    r = np.zeros(12); r[col[f"p_A{h}"]] = 1; r[col[f"p_B{h}"]] = 1
    rows.append(r); lb.append(demand[h]); ub.append(demand[h])
for g in ["A", "B"]:
    for h in range(2):                               # p <= pmax * u   ->  p - pmax*u <= 0
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmax[g]
        rows.append(r); lb.append(-np.inf); ub.append(0.0)
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmin[g]   # p >= pmin * u
        rows.append(r); lb.append(0.0); ub.append(np.inf)
        r = np.zeros(12); r[col[f"s_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -1          # s >= u_h - u_{h-1}
        if h > 0:
            r[col[f"u_{g}{h-1}"]] = 1
        rows.append(r); lb.append(0.0); ub.append(np.inf)
A = np.array(rows); lb = np.array(lb); ub = np.array(ub)
integrality = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])
hi = np.array([pmax["A"], pmax["A"], pmax["B"], pmax["B"]] + [1.0] * 8)

# your solution here
cost = ____
u = ____

q4.check()

In [ ]:
# q4.hint()
# q4.solution()

## Task 5

From the same MILP solution, assign the production plan `p` as a 2×2 DataFrame with index ['A', 'B'] and columns [0, 1] (unit × hour). Check by eye that each column sums to the demand.

Assign the result to `p`.

In [ ]:
# Two units, two hours. Variable order:
#   p_A0, p_A1, p_B0, p_B1   production (MW)      continuous
#   u_A0, u_A1, u_B0, u_B1   on/off               binary
#   s_A0, s_A1, s_B0, s_B1   start-up indicator   binary
names = ["p_A0", "p_A1", "p_B0", "p_B1", "u_A0", "u_A1", "u_B0", "u_B1", "s_A0", "s_A1", "s_B0", "s_B1"]
col = {n: i for i, n in enumerate(names)}
demand = np.array([6.0, 11.0])
mc    = {"A": 10.0, "B": 30.0}     # marginal cost EUR/MWh
oncost = {"A": 5.0, "B": 2.0}      # cost per hour while on
startup = {"A": 20.0, "B": 10.0}   # cost of switching on (both units start OFF)
pmax = {"A": 8.0, "B": 6.0}
pmin = {"A": 2.0, "B": 1.0}

c = np.zeros(12)
for g in ["A", "B"]:
    for h in range(2):
        c[col[f"p_{g}{h}"]] = mc[g]
        c[col[f"u_{g}{h}"]] = oncost[g]
        c[col[f"s_{g}{h}"]] = startup[g]

rows, lb, ub = [], [], []
for h in range(2):                                   # demand: p_A + p_B = demand
    r = np.zeros(12); r[col[f"p_A{h}"]] = 1; r[col[f"p_B{h}"]] = 1
    rows.append(r); lb.append(demand[h]); ub.append(demand[h])
for g in ["A", "B"]:
    for h in range(2):                               # p <= pmax * u   ->  p - pmax*u <= 0
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmax[g]
        rows.append(r); lb.append(-np.inf); ub.append(0.0)
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmin[g]   # p >= pmin * u
        rows.append(r); lb.append(0.0); ub.append(np.inf)
        r = np.zeros(12); r[col[f"s_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -1          # s >= u_h - u_{h-1}
        if h > 0:
            r[col[f"u_{g}{h-1}"]] = 1
        rows.append(r); lb.append(0.0); ub.append(np.inf)
A = np.array(rows); lb = np.array(lb); ub = np.array(ub)
integrality = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])
hi = np.array([pmax["A"], pmax["A"], pmax["B"], pmax["B"]] + [1.0] * 8)

# your solution here
p = ____

q5.check()

In [ ]:
# q5.hint()
# q5.solution()

## Task 6

Solve the LP relaxation of the unit-commitment instance (integrality all zeros). Assign `cost_lp` and the relative gap `gap = cost_mip / cost_lp − 1`.

Assign the result to `cost_lp`, `gap`.

In [ ]:
# Two units, two hours. Variable order:
#   p_A0, p_A1, p_B0, p_B1   production (MW)      continuous
#   u_A0, u_A1, u_B0, u_B1   on/off               binary
#   s_A0, s_A1, s_B0, s_B1   start-up indicator   binary
names = ["p_A0", "p_A1", "p_B0", "p_B1", "u_A0", "u_A1", "u_B0", "u_B1", "s_A0", "s_A1", "s_B0", "s_B1"]
col = {n: i for i, n in enumerate(names)}
demand = np.array([6.0, 11.0])
mc    = {"A": 10.0, "B": 30.0}     # marginal cost EUR/MWh
oncost = {"A": 5.0, "B": 2.0}      # cost per hour while on
startup = {"A": 20.0, "B": 10.0}   # cost of switching on (both units start OFF)
pmax = {"A": 8.0, "B": 6.0}
pmin = {"A": 2.0, "B": 1.0}

c = np.zeros(12)
for g in ["A", "B"]:
    for h in range(2):
        c[col[f"p_{g}{h}"]] = mc[g]
        c[col[f"u_{g}{h}"]] = oncost[g]
        c[col[f"s_{g}{h}"]] = startup[g]

rows, lb, ub = [], [], []
for h in range(2):                                   # demand: p_A + p_B = demand
    r = np.zeros(12); r[col[f"p_A{h}"]] = 1; r[col[f"p_B{h}"]] = 1
    rows.append(r); lb.append(demand[h]); ub.append(demand[h])
for g in ["A", "B"]:
    for h in range(2):                               # p <= pmax * u   ->  p - pmax*u <= 0
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmax[g]
        rows.append(r); lb.append(-np.inf); ub.append(0.0)
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmin[g]   # p >= pmin * u
        rows.append(r); lb.append(0.0); ub.append(np.inf)
        r = np.zeros(12); r[col[f"s_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -1          # s >= u_h - u_{h-1}
        if h > 0:
            r[col[f"u_{g}{h-1}"]] = 1
        rows.append(r); lb.append(0.0); ub.append(np.inf)
A = np.array(rows); lb = np.array(lb); ub = np.array(ub)
integrality = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])
hi = np.array([pmax["A"], pmax["A"], pmax["B"], pmax["B"]] + [1.0] * 8)

# your solution here
cost_lp = ____
gap = ____

q6.check()

In [ ]:
# q6.hint()
# q6.solution()

## Task 7

Rounding the relaxed solution does not work. In the LP relaxation u_B1 is fractional. Force it to 0 (round down) by setting both its lower and upper bound to 0, re-solve as an LP, and assign the resulting `status` (expect 2 = infeasible: unit A alone cannot cover 11 MW).

Assign the result to `status`.

In [ ]:
# Two units, two hours. Variable order:
#   p_A0, p_A1, p_B0, p_B1   production (MW)      continuous
#   u_A0, u_A1, u_B0, u_B1   on/off               binary
#   s_A0, s_A1, s_B0, s_B1   start-up indicator   binary
names = ["p_A0", "p_A1", "p_B0", "p_B1", "u_A0", "u_A1", "u_B0", "u_B1", "s_A0", "s_A1", "s_B0", "s_B1"]
col = {n: i for i, n in enumerate(names)}
demand = np.array([6.0, 11.0])
mc    = {"A": 10.0, "B": 30.0}     # marginal cost EUR/MWh
oncost = {"A": 5.0, "B": 2.0}      # cost per hour while on
startup = {"A": 20.0, "B": 10.0}   # cost of switching on (both units start OFF)
pmax = {"A": 8.0, "B": 6.0}
pmin = {"A": 2.0, "B": 1.0}

c = np.zeros(12)
for g in ["A", "B"]:
    for h in range(2):
        c[col[f"p_{g}{h}"]] = mc[g]
        c[col[f"u_{g}{h}"]] = oncost[g]
        c[col[f"s_{g}{h}"]] = startup[g]

rows, lb, ub = [], [], []
for h in range(2):                                   # demand: p_A + p_B = demand
    r = np.zeros(12); r[col[f"p_A{h}"]] = 1; r[col[f"p_B{h}"]] = 1
    rows.append(r); lb.append(demand[h]); ub.append(demand[h])
for g in ["A", "B"]:
    for h in range(2):                               # p <= pmax * u   ->  p - pmax*u <= 0
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmax[g]
        rows.append(r); lb.append(-np.inf); ub.append(0.0)
        r = np.zeros(12); r[col[f"p_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -pmin[g]   # p >= pmin * u
        rows.append(r); lb.append(0.0); ub.append(np.inf)
        r = np.zeros(12); r[col[f"s_{g}{h}"]] = 1; r[col[f"u_{g}{h}"]] = -1          # s >= u_h - u_{h-1}
        if h > 0:
            r[col[f"u_{g}{h-1}"]] = 1
        rows.append(r); lb.append(0.0); ub.append(np.inf)
A = np.array(rows); lb = np.array(lb); ub = np.array(ub)
integrality = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])
hi = np.array([pmax["A"], pmax["A"], pmax["B"], pmax["B"]] + [1.0] * 8)

# your solution here
status = ____

q7.check()

In [ ]:
# q7.hint()
# q7.solution()

## Task 8

Knapsack. Choose the subset of projects in `proj` that maximises total margin with total capex ≤ `budget`, using milp with all variables binary (minimise −margin). Assign the chosen project names as a sorted list `chosen` and the total margin `best_margin`.

Assign the result to `chosen`, `best_margin`.

In [ ]:
proj = pd.DataFrame({"capex": [40.0, 50.0, 30.0, 60.0], "margin": [10.0, 13.0, 7.0, 15.0]},
                    index=["A", "B", "C", "D"])
budget = 100.0

# your solution here
chosen = ____
best_margin = ____

q8.check()

In [ ]:
# q8.hint()
# q8.solution()

## Task 9

Greedy by margin-per-capex: sort projects by margin/capex descending and add each one if it still fits in the budget. Assign the greedy choice as a sorted list `greedy` and its total margin `greedy_margin`. Compare with the exact answer.

Assign the result to `greedy`, `greedy_margin`.

In [ ]:
proj = pd.DataFrame({"capex": [40.0, 50.0, 30.0, 60.0], "margin": [10.0, 13.0, 7.0, 15.0]},
                    index=["A", "B", "C", "D"])
budget = 100.0

# your solution here
greedy = ____
greedy_margin = ____

q9.check()

In [ ]:
# q9.hint()
# q9.solution()

## Task 10

Two-stage hedge as an LP. Decide the forward volume F now (price K), then in each of 2 equally likely scenarios s buy the shortfall at P_s + c_u or sell the surplus at P_s − c_o. Variables: [F, short_1, long_1, short_2, long_2]. Minimise K·F + Σ_s ½[(P_s + c_u)·short_s − (P_s − c_o)·long_s] subject to F + short_s − long_s = L_s, 0 ≤ F ≤ 20, others ≥ 0. Solve with linprog (method='highs') and assign `F_star`.

Assign the result to `F_star`.

In [ ]:
L = np.array([10.0, 14.0])
P = np.array([100.0, 120.0])
K = 105.0
c_u, c_o = 25.0, 12.0

# your solution here
F_star = ____

q10.check()

In [ ]:
# q10.hint()
# q10.solution()

## Task 11

`bumpy(x) = (x − 2)² + 3·sin(3x)` has several local minima. Run `minimize` (Nelder-Mead) from x0 = 0 and `differential_evolution` on bounds [(−2, 6)] with seed=0. Assign the two minimisers `x_local` and `x_global` (floats). Are they the same?

Assign the result to `x_local`, `x_global`.

In [ ]:
def bumpy(x_):
    x_ = np.asarray(x_).ravel()[0]
    return (x_ - 2) ** 2 + 3 * np.sin(3 * x_)

# your solution here
x_local = ____
x_global = ____

q11.check()

In [ ]:
# q11.hint()
# q11.solution()

## Task 12

Real data, merit order by hand. For 2023-01-24, the demand in GW is `demand_gw` (24 values). Three units: nuclear 10 GW at 12 EUR/MWh, ccgt 20 GW at 80, peaker 10 GW at 160. Dispatch each hour greedily (cheapest first up to its capacity) and assign the total energy cost for the day in EUR (GW × 1000 = MW, hourly steps so MW·h = MWh).

Assign the result to `answer`.

In [ ]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
demand_gw = df.loc["2023-01-24", "consumption_mwh"].values / 1000
cap = np.array([10.0, 20.0, 10.0])
mc = np.array([12.0, 80.0, 160.0])

# your solution here
answer = ____

q12.check()

In [ ]:
# q12.hint()
# q12.solution()

---
Done? Re-open the cheat sheet for anything you had to look up, then try the next `_tests` notebook.